# Beskattningskurva för delägarbeskattade samfälligheter

Verktyget räknar ut vad varje delägarfastighet ska ta upp i deklarationen, och visar
**vid vilket resultat för föreningen** delägarna behöver börja deklarera.

> ⚠️ Sammanfattning skriven av en språkmodell (AI). Enbart upplysning, inte
> skatterådgivning. Se [Skatteverket](https://www.skatteverket.se) för aktuella regler.

## Så kör du – fyra steg

**1. Lägg in föreningens matrikel**
Öppna mappen **data** i fillistan till vänster. Ersätt `andelar.csv` med er egen fil
(dra in den från datorn) eller dubbelklicka på filen och skriv in era andelar,
**ett andelstal per rad**, till exempel `0,1679`. Spara med **Ctrl+S** (Mac: **⌘+S**).

**2. Fyll i bokslutets siffror**
Klicka i den grå rutan under rubriken **1. Inställningar** nedan och ändra
siffrorna efter `"intakter"` och `"kostnader"` (och vid behov de andra raderna) till
värdena i föreningens resultaträkning. Ändra också `AR` till rätt år.

**3. Kör allt**
Välj menyn **Run → Run All Cells** högst upp. Vänta några sekunder.
Första gången kan det ta upp till en minut.

**4. Läs resultatet**
Scrolla ner. Du ser beskattningskurvan, vilka fastigheter som ska deklarera och
en uppskattning av den extra skatten. Utskicket till delägarna sparas i mappen
**utdata**. Högerklicka på filen och välj **Download** för att spara den på datorn.

Har du ändrat något och vill räkna om? Gör bara steg 3 igen.

*Mer om reglerna finns i [README på GitHub](https://github.com/HuggeK/beskattningFVOF#readme).*


## 1. Inställningar

Ändra bara siffrorna till höger om `=` och `:`. Texten efter `#` är förklaringar.

In [ ]:
# ============================================================
# INSTÄLLNINGAR – ändra här
# ============================================================
AR = 2026                        # Räkenskapsåret underlaget gäller

# Siffror från resultaträkningen i bokslutet (EXEMPEL – ersätt med era egna).
# Intäkter och kostnader krävs: näringsfastigheter ska ta upp sin andel av dem var för sig.
# Belopp kan skrivas som tal (95000) eller text ("95 000,50").
BOKSLUT = {
    "intakter": 95000,           # Summa rörelseintäkter (fiskekort, bidrag m.m.)
    "kostnader": 58000,          # Summa rörelsekostnader, inklusive avskrivningar
    "ranteintakter": 400,        # Finansiella intäkter (ränta, utdelning) – behövs för KU73
    "ranteutgifter": 0,          # Finansiella kostnader
    "varav_uttaxering": 8500,    # Avgifter från delägarna som ingår i intäkterna – räknas bort
}

ANDELSFIL = "data/andelar.csv"
UTDATAMAPP = "utdata"

# "andel"  = andelstalen summerar till 1
# "mantal" = andelstalen är mantal och normaliseras mot summan av alla mantal
ANDELSTYP = "andel"

# Beloppsgränser – kontrollera hos Skatteverket varje år
GRANS_NARING = 1500              # näringsfastighet: netto under detta behöver inte tas upp
GRANS_RANTA_PRIVAT = 1500        # privatbostad: ränta/utdelning (ruta 7.2) och KU73
SCHABLON_PRIVAT = 40000          # privatbostad: schablonavdrag per fastighet och år
SCHABLON_PRIVAT_PROCENT = "0.20" # + 20 % av intäkten

# Uppskattad marginalskatt på det som tas upp i näringsverksamhet – ett spann, inte ett facit.
#   lag    ≈ aktiebolag (bolagsskatt 20,6 %). Den som har underskott i sin NE betalar 0 kr just i år.
#   trolig ≈ passiv/aktiv näringsverksamhet: kommunalskatt ~32 % + särskild löneskatt 24,26 %
#            eller egenavgifter 28,97 %, efter schablonavdrag ≈ 50 %
#   hog    ≈ som trolig + statlig inkomstskatt 20 % över skiktgränsen ≈ 65 %
SKATTESATS = {"lag": 0.206, "trolig": 0.50, "hog": 0.65}
SKATTESATS_KAPITAL = 0.30        # privatbostad: inkomst av kapital

# Beskattningskurvan (simulering)
SIM_MAX_RESULTAT = 150000        # hur långt kurvan ska gå (kr)
SCENARIER = [10000, 25000, 50000, 100000]   # resultat att jämföra, t.ex. budgetalternativ


## 2. Läs in andelar och räkna fram resultatet

In [ ]:
import csv
import math
from collections import Counter
from decimal import Decimal
from fractions import Fraction
from pathlib import Path


def tal(text):
    """Tolkar '0,1679', '0.1679', '5/64', '"1 234,50"' eller '' som ett exakt bråk."""
    s = (text or "").strip().strip('"').strip("'").replace(" ", "").replace("\u00a0", "").replace(",", ".")
    if not s:
        return Fraction(0)
    if "/" in s:
        t, n = s.split("/", 1)
        return Fraction(Decimal(t)) / Fraction(Decimal(n))
    return Fraction(Decimal(s))


def ar_tal(s):
    try:
        tal(s)
        return True
    except Exception:
        return False


def fmt(x):
    return f"{int(math.floor(x + Fraction(1, 2))):,}".replace(",", " ")


def las_tabell(sokvag):
    """Läser semikolon- eller tab-separerad fil, eller en enkel lista med ett tal per rad."""
    with open(sokvag, encoding="utf-8-sig") as f:
        rader = [r.rstrip("\r\n") for r in f if r.strip()]
    delim = ";" if ";" in rader[0] else ("\t" if "\t" in rader[0] else None)
    if delim is None:                                   # enkel lista
        return [{"andel": r} for r in rader if ar_tal(r)]
    celler = [[c.strip().strip('"') for c in r.split(delim)] for r in rader]
    if ar_tal(celler[0][0]) or (len(celler[0]) > 1 and ar_tal(celler[0][1])):   # utan rubrikrad
        return [{"andel": c[0]} if ar_tal(c[0]) else {"fastighet": c[0], "andel": c[1]} for c in celler]
    rubriker = [h.lower() for h in celler[0]]
    return [dict(zip(rubriker, c + [""] * len(rubriker))) for c in celler[1:]]


# ---------- Andelar ----------
# Frivillig kolumn 'typ': naring / privat. Tom = okänd (räknas som näring i kurvan, det strängaste fallet).
TYPER = {"naring": "naring", "näring": "naring", "n": "naring", "naringsfastighet": "naring",
         "näringsfastighet": "naring", "privat": "privat", "p": "privat", "privatbostad": "privat",
         "privatbostadsfastighet": "privat"}
andelar = []
for nr, rad in enumerate(las_tabell(ANDELSFIL), 1):
    andelar.append({
        "fastighet": (rad.get("fastighet") or "").strip() or f"Fastighet {nr:02d}",
        "andelstal": tal(rad["andel"]),
        "typ": TYPER.get((rad.get("typ") or "").strip().lower(), ""),
    })
summa = sum(a["andelstal"] for a in andelar)
for a in andelar:
    a["andel"] = a["andelstal"] / summa if ANDELSTYP == "mantal" else a["andelstal"]
print(f"{len(andelar)} delägarfastigheter, summa andelstal {float(summa):.6f}")
antal_typ = Counter(a["typ"] or "okänd" for a in andelar)
print("Fastighetstyp: " + ", ".join(f"{k} {v} st" for k, v in sorted(antal_typ.items())))
if ANDELSTYP == "andel" and abs(summa - 1) > Fraction(1, 10**6):
    print("VARNING: andelarna summerar inte till 1 – kontrollera filen eller sätt ANDELSTYP = 'mantal'.")

# ---------- Bokslut ----------
b = {k: ("" if v is None else str(v)) for k, v in BOKSLUT.items()}
for nyckel in ("intakter", "kostnader"):
    if not b.get(nyckel, "").strip():
        raise ValueError(f"Fyll i '{nyckel}' i BOKSLUT – intäkter och kostnader ska tas upp var för sig.")
RANTA = tal(b.get("ranteintakter"))
RANTEUTGIFTER = tal(b.get("ranteutgifter"))
INTAKTER = tal(b["intakter"]) - tal(b.get("varav_uttaxering")) + RANTA
KOSTNADER = tal(b["kostnader"]) + RANTEUTGIFTER
UPPLATELSE = INTAKTER - RANTA          # förenkling: alla rörelseintäkter räknas som upplåtelse för privatbostad
RESULTAT = INTAKTER - KOSTNADER

print(f"\nFöreningen {AR}:  intäkter {fmt(INTAKTER)} kr  –  kostnader {fmt(KOSTNADER)} kr  "
      f"=  resultat {fmt(RESULTAT)} kr")
if tal(b.get("varav_uttaxering")):
    print(f"  (uttaxering {fmt(tal(b['varav_uttaxering']))} kr är borträknad ur intäkterna)")


## 3. Beskattningskurvan

Kärnan i verktyget. Kurvan visar **vid vilket resultat för föreningen** delägare med näringsfastighet måste börja deklarera, och hur mycket som då tas upp. Använd den för att planera, till exempel inför ett år med stora utgifter eller vid budgetering av fiskekortspriser. Ändra `SCENARIER` i inställningarna för att jämföra olika utfall.

- **Grönt fält:** ingen behöver deklarera.
- **Orange fält:** underskott, alla näringsfastigheter *får* ta upp sin andel.
- **Nedre diagrammet:** för varje fastighet visas resultatet där just den går över gränsen.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Deklarationsgräns per fastighet: minsta resultat för föreningen där andel × resultat ≥ 1 500 kr
# Privatbostäder har andra regler (schablonavdrag) och räknas inte i kurvan.
for a in andelar:
    a["grans"] = (math.ceil(Fraction(GRANS_NARING) / a["andel"])
                  if a["andel"] > 0 and a["typ"] != "privat" else None)
granser = sorted(a["grans"] for a in andelar if a["grans"])
antal_privat = sum(1 for a in andelar if a["typ"] == "privat")
if antal_privat:
    print(f"{antal_privat} fastigheter är angivna som privatbostäder och räknas inte i kurvan "
          f"(de deklarerar bara om andelen av intäkterna överstiger ca 50 000 kr).\n")

def antal_ska(res):
    return sum(1 for g in granser if res >= g)

def summa_upptas(res):
    return sum(float(a["andel"] * res) for a in andelar if a["grans"] and res >= a["grans"])

def sammanfatta(res):
    n = antal_ska(res)
    s = summa_upptas(res)
    andel = 100 * s / res if res > 0 else 0
    return n, len(andelar) - n, s, andel

# --- Scenarier ---
arets = RESULTAT if RESULTAT is not None else None
lista = ([("Årets resultat " + str(AR), arets)] if arets is not None else []) + [(f"Scenario {i}", x) for i, x in enumerate(SCENARIER, 1)]
print(f"Ingen näringsfastighet behöver deklarera om föreningens resultat är under {fmt(granser[0])} kr.\n")
print(f"{'':<22} {'Resultat':>10} {'Ska deklarera':>14} {'Under gränsen':>14} {'Tas upp totalt':>15} "
      f"{'Extra skatt totalt, trolig (spann)':>36}")
for namn, res in lista:
    if res < 0:
        print(f"{namn:<22} {fmt(res):>10}   underskott – alla näringsfastigheter FÅR ta upp sin andel")
        continue
    n, under, s, pct = sammanfatta(res)
    skatt = {k: Fraction(s) * Fraction(str(v)) for k, v in SKATTESATS.items()}
    print(f"{namn:<22} {fmt(res):>10} {n:>14} {under:>14} {fmt(Fraction(s)):>12} kr "
          f"{fmt(skatt['trolig']):>14} kr ({fmt(skatt['lag'])}–{fmt(skatt['hog'])} kr)")

# --- Planering: brytpunkterna ---
unika = sorted(set(granser))
print("\nBrytpunkter – håll resultatet UNDER en brytpunkt så slipper fler ta upp sin andel:")
print(f"  {'Föreningens resultat':<24} {'Ska deklarera':>13}   Tillkommer vid intervallets början")
starter = [0] + unika[:20]
for i, start in enumerate(starter):
    slut = f"{fmt(starter[i + 1] - 1)} kr" if i + 1 < len(starter) else "och uppåt"
    nya = ", ".join(a["fastighet"] for a in andelar if a["grans"] == start) if start else ""
    print(f"  {fmt(start) + ' – ' + slut:<24} {antal_ska(start):>13}   {('+ ' + nya) if nya else ''}")
if arets is not None and arets > 0:
    under = [g for g in unika if g <= arets]
    over = [g for g in unika if g > arets]
    if under:
        g = under[-1]
        print(f"\nÅrets resultat {fmt(arets)} kr ligger {fmt(arets - g)} kr över brytpunkten {fmt(g)} kr. "
              f"Under {fmt(g)} kr hade {antal_ska(arets) - antal_ska(g - 1)} färre behövt deklarera.")
    if over:
        print(f"Nästa brytpunkt är {fmt(over[0])} kr ({fmt(over[0] - arets)} kr bort) – då tillkommer "
              f"{antal_ska(over[0]) - antal_ska(arets)} till.")

# --- Diagram ---
kr_fmt = FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " "))
xmax = max(SIM_MAX_RESULTAT, int(arets * Fraction(6, 5)) if arets and arets > 0 else 0)
xmin = min(-0.12 * xmax, float(arets) * 1.15 if arets is not None and arets < 0 else 0)
xs = sorted({0, xmax} | {g for g in granser if g <= xmax})
fig, (ax, bx) = plt.subplots(2, 1, figsize=(14, 13), dpi=100, sharex=True,
                             gridspec_kw={"height_ratios": [1, 1.25]})

# (1) Kurvan: antal som ska deklarera och summan som tas upp
ax.axvspan(xmin, 0, color="tab:orange", alpha=0.12)
ax.axvspan(0, granser[0], color="tab:green", alpha=0.12)
ax.text(xmin / 2, 0.5, "Underskott:\nalla FÅR\nta upp andelen", ha="center", va="center",
        transform=ax.get_xaxis_transform(), fontsize=9, color="tab:orange")
ax.text(granser[0] / 2, 0.5, "Ingen\nbehöver\ndeklarera", ha="center", va="center",
        transform=ax.get_xaxis_transform(), fontsize=9, color="tab:green")
ax.step(xs, [antal_ska(x) for x in xs], where="post", color="tab:red", linewidth=2,
        label="Antal som SKA deklarera (vänster axel)")
ax.set_ylabel("Antal näringsfastigheter som ska deklarera", color="tab:red")
ax.set_ylim(0, max(antal_ska(xmax) * 1.15, 1))
ax.yaxis.get_major_locator().set_params(integer=True)
ax2 = ax.twinx()
xf = sorted(set(xs) | set(range(0, xmax + 1, max(1, xmax // 600))))
ax2.plot(xf, [summa_upptas(x) for x in xf], color="tab:blue", linewidth=1.5,
         label="Summa som tas upp i deklarationer (höger axel)")
ax2.plot([0, xmax], [0, xmax], color="tab:blue", linestyle=":", linewidth=1, alpha=0.5,
         label="Hela resultatet (om alla deklarerade)")
# Spann för total extra skatt för medlemmarna vid varje resultatnivå
skatt_lag = [summa_upptas(x) * SKATTESATS["lag"] for x in xf]
skatt_trolig = [summa_upptas(x) * SKATTESATS["trolig"] for x in xf]
skatt_hog = [summa_upptas(x) * SKATTESATS["hog"] for x in xf]
ax2.fill_between(xf, skatt_lag, skatt_hog, color="tab:purple", alpha=0.18, step=None,
                 label=f"Extra skatt totalt, spann ({SKATTESATS['lag'] * 100:.0f}–{SKATTESATS['hog'] * 100:.0f} %)")
ax2.plot(xf, skatt_trolig, color="tab:purple", linewidth=1.5,
         label=f"Extra skatt totalt, trolig ({SKATTESATS['trolig'] * 100:.0f} %)")
if arets is not None and 0 < arets <= xmax:
    s_ar = summa_upptas(arets)
    ax2.annotate(f"Vid {fmt(arets)} kr: extra skatt ca {fmt(Fraction(s_ar * SKATTESATS['trolig']).limit_denominator())} kr\n"
                 f"(spann {fmt(Fraction(s_ar * SKATTESATS['lag']).limit_denominator())}–"
                 f"{fmt(Fraction(s_ar * SKATTESATS['hog']).limit_denominator())} kr)",
                 (float(arets), s_ar * SKATTESATS["trolig"]), xytext=(float(arets) + xmax * 0.05, xmax * 0.55),
                 fontsize=8.5, color="tab:purple", arrowprops=dict(arrowstyle="->", color="tab:purple"),
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="tab:purple", alpha=0.9))
ax2.set_ylabel("Kronor: summa som tas upp / extra skatt", color="tab:blue")
ax2.set_ylim(0, xmax * 1.15)
ax2.yaxis.set_major_formatter(kr_fmt)
# Streckade linjer vid varje brytpunkt – här blir trappan skarp
senast_etikett = -math.inf
for g in unika:
    if g > xmax:
        break
    ax.axvline(g, color="tab:red", linestyle=(0, (4, 3)), linewidth=0.9, alpha=0.55, zorder=1)
    bx.axvline(g, color="tab:red", linestyle=(0, (4, 3)), linewidth=0.6, alpha=0.3, zorder=0)
    if g - senast_etikett > xmax * 0.035:
        ax.text(g, 1.01, f"{fmt(g)} kr  {antal_ska(g - 1)}→{antal_ska(g)}", transform=ax.get_xaxis_transform(),
                rotation=90, ha="center", va="bottom", fontsize=7, color="tab:red")
        senast_etikett = g
ax.scatter([g for g in unika if g <= xmax], [antal_ska(g) for g in unika if g <= xmax],
           color="tab:red", s=14, zorder=5)

forsta = min(andelar, key=lambda a: a["grans"] or math.inf)
ax.annotate(f"Första som måste deklarera:\n{forsta['fastighet']} från {fmt(granser[0])} kr",
            (granser[0], 1), xytext=(granser[0] + xmax * 0.03, antal_ska(xmax) * 0.45),
            fontsize=9, arrowprops=dict(arrowstyle="->", color="tab:red"),
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="tab:red", alpha=0.9))
for namn, res in lista:
    if xmin <= res <= xmax:
        stil = dict(color="black", linestyle="--") if namn.startswith("Årets") else dict(color="gray", linestyle=":")
        ax.axvline(res, linewidth=1.2, **stil)
        ax.text(res, 0.98, f" {namn}: {fmt(res)} kr", transform=ax.get_xaxis_transform(),
                rotation=90, ha="right", va="top", fontsize=8, color=stil["color"])
ax.set_xlim(xmin, xmax)
ax.xaxis.set_major_formatter(kr_fmt)
ax.set_xlabel("Föreningens TOTALA resultat för året (kr)")
ax.grid(alpha=0.3)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=9)
ax.set_title(f"Beskattningskurvan: när måste delägare med näringsfastighet deklarera? (gräns {GRANS_NARING} kr)\n"
             "Röda streckade linjer = brytpunkter: håll resultatet strax under dem så slipper fler ta upp sin andel",
             pad=78)

# (2) Per fastighet: från vilket resultat ska just den fastigheten deklarera?
sort = sorted([a for a in andelar if a["grans"]], key=lambda a: a["grans"])
visa = sort[:30]
for i, a in enumerate(visa):
    g = a["grans"]
    bx.barh(i, min(g, xmax), color="lightgray")
    if g < xmax:
        bx.barh(i, xmax - g, left=g, color="tab:red", alpha=0.75)
        bx.text(g, i, f" {fmt(g)} kr", va="center", fontsize=7, color="white" if g < xmax * 0.8 else "black")
    else:
        bx.text(xmax, i, f"  > {fmt(xmax)} kr", va="center", ha="right", fontsize=7)
bx.set_yticks(range(len(visa)))
bx.set_yticklabels([f"{a['fastighet']} ({float(a['andel'] * 100):.2f} %)" for a in visa], fontsize=8)
bx.invert_yaxis()
for namn, res in lista:
    if 0 <= res <= xmax:
        bx.axvline(res, color="black" if namn.startswith("Årets") else "gray",
                   linestyle="--" if namn.startswith("Årets") else ":", linewidth=1.2)
bx.set_xlim(xmin, xmax)                 # samma x-axel som kurvan ovan
bx.axvspan(xmin, 0, color="tab:orange", alpha=0.12, zorder=0)
bx.axvspan(0, granser[0], color="tab:green", alpha=0.12, zorder=0)
ax.tick_params(labelbottom=True)       # visa x-värden även på övre diagrammet
bx.xaxis.set_major_formatter(kr_fmt)
bx.set_xlabel("Föreningens TOTALA resultat för året (kr)")
rest = len(sort) - len(visa)
if antal_privat:
    bx.text(0.99, 0.01, f"{antal_privat} privatbostäder ingår inte", transform=bx.transAxes,
            ha="right", va="bottom", fontsize=8, color="tab:blue")
bx.set_title("Per fastighet: vid vilket TOTALT resultat för föreningen måste just den fastigheten deklarera?\n"
             f"Grått = fastigheten är under gränsen, rött = fastigheten ska deklarera. De {len(visa)} största andelarna"
             + (f" (övriga {rest} först över {fmt(visa[-1]['grans'])} kr)" if rest else ""), fontsize=10)
bx.grid(alpha=0.3, axis="x")

# Kolumn till höger: vad varje fastighet skulle deklarera vid årets resultat (andel × resultat)
if arets is not None:
    from matplotlib.transforms import blended_transform_factory
    hoger = blended_transform_factory(bx.transAxes, bx.transData)
    bx.text(1.01, -1.2, f"Deklarerar vid\n{fmt(arets)} kr", transform=hoger, fontsize=8,
            fontweight="bold", va="bottom", ha="left")
    for i, a in enumerate(visa):
        belopp = a["andel"] * arets
        if belopp >= GRANS_NARING:
            text, farg = f"{fmt(belopp)} kr – SKA", "tab:red"
        elif belopp < 0:
            text, farg = f"{fmt(belopp)} kr (får)", "tab:orange"
        else:
            text, farg = f"{fmt(belopp)} kr – behöver inte", "gray"
        bx.text(1.01, i, text, transform=hoger, va="center", ha="left", fontsize=7.5, color=farg,
                fontweight="bold" if farg == "tab:red" else "normal")

fig.tight_layout()
Path(UTDATAMAPP).mkdir(exist_ok=True)
fig.savefig(Path(UTDATAMAPP) / "beskattningskurva.png", bbox_inches="tight")
plt.show()


## 4. Underlag per fastighet för valt år

Varje rad visar utfallet både om fastigheten är en näringsfastighet och om den är en privatbostad. Kolumnen `naring_deklarera_fran_resultat` i utdatafilen anger från vilket resultat för föreningen fastigheten måste deklarera.

> Sammanfattning av en språkmodell – enbart upplysning. Se [Skatteverket](https://www.skatteverket.se) för aktuella skatteregler.

In [ ]:
procent = Fraction(SCHABLON_PRIVAT_PROCENT)
rader = []
for a in andelar:
    x = a["andel"]
    intakt, kostnad = INTAKTER * x, KOSTNADER * x
    netto = intakt - kostnad
    om_naring = ("SKA deklarera" if netto >= GRANS_NARING
                 else "FÅR välja (underskott)" if netto < 0
                 else "Behöver inte")
    u = UPPLATELSE * x
    ruta_7_3 = max(Fraction(0), u - SCHABLON_PRIVAT - procent * u)
    ruta_7_2 = max(Fraction(0), RANTA * x - GRANS_RANTA_PRIVAT)
    om_privat = "Inget att deklarera" if ruta_7_3 == 0 and ruta_7_2 == 0 else "Deklarera, se rutor"
    if a["typ"] == "naring":
        status = om_naring
    elif a["typ"] == "privat":
        status = om_privat
    else:
        status = f"Näring: {om_naring} / Privat: {om_privat}"
    rader.append({
        "fastighet": a["fastighet"], "typ": a["typ"] or "okänd", "status": status, "andel": x,
        "intakter": intakt, "kostnader": kostnad, "netto": netto,
        "om_naringsfastighet": om_naring, "om_privatbostad": om_privat,
        "privat_ruta_7_3": ruta_7_3, "privat_ruta_7_2": ruta_7_2, "privat_ruta_8_1": RANTEUTGIFTER * x,
        "ku73_om_privat": ruta_7_2,
        "naring_deklarera_fran_resultat": a.get("grans") or "",
    })

print(f"{'Fastighet':<16} {'Typ':<7} {'Andel %':>8} {'Intäkter':>9} {'Kostnader':>10} {'Netto':>8}  Status")
for r in rader:
    print(f"{r['fastighet'][:16]:<16} {r['typ']:<7} {float(r['andel'] * 100):>8.3f} {fmt(r['intakter']):>9} "
          f"{fmt(r['kostnader']):>10} {fmt(r['netto']):>8}  {r['status']}")

print("\nSammanställning:")
for (typ, status), antal in sorted(Counter((r["typ"], r["status"]) for r in rader).items()):
    print(f"  {typ:<7} {status:<60} {antal:>3} st")

storst = max(rader, key=lambda r: r["andel"])
if any(r["privat_ruta_7_2"] > 0 for r in rader):
    print("\nKU73 kan behövas för fastigheter som är privatbostäder:")
    for r in rader:
        if r["privat_ruta_7_2"] > 0:
            print(f"  {r['fastighet']}: {fmt(r['ku73_om_privat'])} kr (delas lika om fastigheten har flera ägare)")
else:
    print(f"\nKU73 behövs inte: största andel av räntan är {fmt(RANTA * storst['andel'])} kr "
          f"(gräns {GRANS_RANTA_PRIVAT} kr).")

# Export – semikolon och decimalkomma, öppnas direkt i svensk Excel/LibreOffice
Path(UTDATAMAPP).mkdir(exist_ok=True)
utfil = Path(UTDATAMAPP) / f"underlag_{AR}.csv"
with open(utfil, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(list(rader[0]))
    for r in rader:
        w.writerow([f"{float(v):.6f}".replace(".", ",") if k == "andel"
                    else fmt(v).replace(" ", "") if isinstance(v, Fraction) else v
                    for k, v in r.items()])
print(f"\nSparat: {utfil}")
print("Sammanfattning av en språkmodell – enbart upplysning. Se Skatteverket för aktuella regler.")


In [ ]:
sorterade = sorted(rader, key=lambda r: r["netto"], reverse=True)
farger = ["tab:blue" if r["typ"] == "privat" else "tab:red" if r["netto"] >= GRANS_NARING
          else "tab:orange" if r["netto"] < 0 else "lightgray" for r in sorterade]
fig, ax = plt.subplots(figsize=(14, 4.5), dpi=100)
ax.bar(range(len(sorterade)), [float(r["netto"]) for r in sorterade], color=farger)
ax.axhline(GRANS_NARING, color="tab:red", linestyle="--", linewidth=1, label=f"Gräns {GRANS_NARING} kr")
ax.axhline(0, color="black", linewidth=0.5)
ax.yaxis.set_major_formatter(kr_fmt)
ax.set_xlabel("Fastigheter, sorterade efter netto")
ax.set_ylabel("Andelens netto (kr)")
ax.set_title(f"{AR}: rött = ska deklarera, grått = behöver inte, orange = får välja (underskott), blått = privatbostad")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()


## 5. Uppskattad extra skatt per fastighet och totalt

Summerar vad varje fastighet ska ta upp och uppskattar skatten som ett **spann** med satserna i `SKATTESATS`. Det är en grov uppskattning, inte ett facit.

In [ ]:
# ---------- Uppskattad extra skatt per fastighet och totalt ----------
satser = {k: Fraction(str(v)) for k, v in SKATTESATS.items()}
kap = Fraction(str(SKATTESATS_KAPITAL))
skatt_rader = []
for r, a in zip(rader, andelar):
    if a["typ"] == "privat":
        naring, kapital, underskott = Fraction(0), r["privat_ruta_7_3"] + r["privat_ruta_7_2"], Fraction(0)
    else:                                   # näring eller okänd (räknas som näring = det strängaste fallet)
        naring = r["netto"] if r["netto"] >= GRANS_NARING else Fraction(0)
        underskott = r["netto"] if r["netto"] < 0 else Fraction(0)
        kapital = Fraction(0)
    rad = {"fastighet": a["fastighet"], "ta_upp": naring + kapital, "underskott": underskott}
    for k, sats in satser.items():
        rad["skatt_" + k] = naring * sats + kapital * kap
        rad["sankning_" + k] = -underskott * sats
    skatt_rader.append(rad)

berorda = [r for r in skatt_rader if r["ta_upp"] > 0]
print(f"Skattesatser i spannet: låg {float(satser['lag']) * 100:.1f} %, trolig {float(satser['trolig']) * 100:.0f} %, "
      f"hög {float(satser['hog']) * 100:.0f} % (näring), kapital {float(kap) * 100:.0f} %.\n")
if berorda:
    print(f"{'Fastighet':<18} {'Ska ta upp':>11} {'Skatt trolig':>13}   Spann")
    for r in sorted(berorda, key=lambda r: -r["skatt_trolig"]):
        print(f"{r['fastighet'][:18]:<18} {fmt(r['ta_upp']):>8} kr {fmt(r['skatt_trolig']):>10} kr   "
              f"{fmt(r['skatt_lag'])}–{fmt(r['skatt_hog'])} kr")
else:
    print("Ingen fastighet behöver ta upp något i år.")

tot = {k: sum(r["skatt_" + k] for r in skatt_rader) for k in satser}
print(f"\nAlla medlemmar tillsammans: ca {fmt(tot['trolig'])} kr i extra skatt "
      f"(spann {fmt(tot['lag'])}–{fmt(tot['hog'])} kr).")
if berorda:
    print(f"  {len(berorda)} av {len(skatt_rader)} fastigheter berörs → i snitt {fmt(tot['trolig'] / len(berorda))} kr "
          f"per berörd fastighet ({fmt(tot['lag'] / len(berorda))}–{fmt(tot['hog'] / len(berorda))} kr).")
if any(r["underskott"] < 0 for r in skatt_rader):
    sank = {k: sum(r["sankning_" + k] for r in skatt_rader) for k in satser}
    print(f"\nUnderskott: om alla med näringsfastighet tar upp sin andel kan deras skatt minska med ca "
          f"{fmt(sank['trolig'])} kr totalt ({fmt(sank['lag'])}–{fmt(sank['hog'])} kr) – förutsatt att de har "
          f"överskott att kvitta mot.")
print("\nOBS: grov uppskattning. Den verkliga skatten beror på ägarnas övriga inkomster, ålder, om verksamheten "
      "är aktiv/passiv och om ägaren är ett bolag. Okänd fastighetstyp räknas som näring.")
